In [2]:
import requests

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

In [4]:
# Download 125 Pages

for i in range(1, 126):
    url = f"https://www.flipkart.com/search?q=smartphones&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=on&as=off&page={i}"
    response = requests.get(url, headers=headers)

    with open(f"Flipkart_Phones/page={i}.html", "w", encoding="utf-8") as f:
        f.write(response.text)
        print(f"Page {i} Downloaded Successfully")

Page 1 Downloaded Successfully
Page 2 Downloaded Successfully
Page 3 Downloaded Successfully
Page 4 Downloaded Successfully
Page 5 Downloaded Successfully
Page 6 Downloaded Successfully
Page 7 Downloaded Successfully
Page 8 Downloaded Successfully
Page 9 Downloaded Successfully
Page 10 Downloaded Successfully
Page 11 Downloaded Successfully
Page 12 Downloaded Successfully
Page 13 Downloaded Successfully
Page 14 Downloaded Successfully
Page 15 Downloaded Successfully
Page 16 Downloaded Successfully
Page 17 Downloaded Successfully
Page 18 Downloaded Successfully
Page 19 Downloaded Successfully
Page 20 Downloaded Successfully
Page 21 Downloaded Successfully
Page 22 Downloaded Successfully
Page 23 Downloaded Successfully
Page 24 Downloaded Successfully
Page 25 Downloaded Successfully
Page 26 Downloaded Successfully
Page 27 Downloaded Successfully
Page 28 Downloaded Successfully
Page 29 Downloaded Successfully
Page 30 Downloaded Successfully
Page 31 Downloaded Successfully
Page 32 Downloade

In [8]:
from bs4 import BeautifulSoup
import re
import pandas as pd

In [16]:
import re
from bs4 import BeautifulSoup

items = []

for page in range(1, 126):
    try:
        with open(f"Flipkart_Phones/page={page}.html", "r", encoding="utf-8") as f:
            content = f.read()
            
            soup = BeautifulSoup(content, "html.parser")
            phones = soup.select("div.ZFwe0M")
            print(f"Processing Page {page}")
            
            for phone in phones:
                
                # PRODUCT NAME
                name_tag = phone.select_one("div.RG5Slk")
                product_name = name_tag.get_text(strip=True) if name_tag else "N/A"
                
                # BRAND
                brand = product_name.split()[0] if product_name != "N/A" else "N/A"

                # PRICE
                price_tag = phone.select_one("div.hZ3P6w")
                if price_tag:
                    price = price_tag.text.replace("₹", "").replace(",", "").strip()
                else:
                    price = None
                    
                # ORIGINAL PRICE
                original_price_tag = phone.select_one("div.kRYCnD")
                if original_price_tag:
                    original_price = original_price_tag.text.replace("₹", "").replace(",", "").strip()
                else:
                    original_price = None
                    
                # DISCOUNT %
                discount_tag = phone.select_one("div.HQe8jr span")
                if discount_tag:
                    discount = discount_tag.text.replace("% off", "").replace("%", "").strip()
                else:
                    discount = None
                    
                # RATING
                rating_tag = phone.select_one("div.MKiFS6")
                rating = rating_tag.text.strip()[:3] if rating_tag else None
                
                # RATINGS / REVIEWS
                ratings_count = None
                reviews_count = None
                
                review_tag = phone.select_one("span.PvbNMB")
                if review_tag:
                    text = review_tag.get_text(" ", strip=True)
                    match = re.findall(r'([\d,]+)', text)
                    if len(match) >= 2:
                        ratings_count = match[0].replace(",", "")
                        reviews_count = match[1].replace(",", "")
                
                # SPECIFICATIONS
                specs = phone.select("div.CMXw7N ul.HwRTzP li.DTBslk")
                
                ram = None
                storage = None
                rear_camera = None
                front_camera = None
                battery = None
                
                if len(specs) > 0:
                    # 1. RAM + STORAGE (Index 0)
                    first_spec = specs[0].text
                    parts = first_spec.split("|")
                    if len(parts) >= 2:
                        ram_match = re.search(r'(\d+)\s*GB', parts[0])
                        storage_match = re.search(r'(\d+)\s*GB', parts[1])
                        ram = ram_match.group(1) if ram_match else None
                        storage = storage_match.group(1) if storage_match else None
                
                if len(specs) > 2:
                    # 2. CAMERAS (Index 2) 
                    camera_spec = specs[2].text
                    camera_parts = camera_spec.split("|")
                    
                    rear_camera = camera_parts[0].strip() if len(camera_parts) > 0 else None
                    front_camera = camera_parts[1].strip() if len(camera_parts) > 1 else None

                if len(specs) > 3:
                    # 3. BATTERY (Index 3)
                    battery_match = re.search(r'(\d+)\s*mAh', specs[3].text)
                    battery = battery_match.group(1) if battery_match else None
                
                items.append([
                    product_name,
                    brand,
                    price,
                    original_price,
                    discount,
                    rating,
                    ratings_count,
                    reviews_count,
                    ram,
                    storage,
                    rear_camera,
                    front_camera,
                    battery
                ])
                
    except Exception as e:
        print(f"Error on page {page}: {e}")  

Processing Page 1
Processing Page 2
Processing Page 3
Processing Page 4
Processing Page 5
Processing Page 6
Processing Page 7
Processing Page 8
Processing Page 9
Processing Page 10
Processing Page 11
Processing Page 12
Processing Page 13
Processing Page 14
Processing Page 15
Processing Page 16
Processing Page 17
Processing Page 18
Processing Page 19
Processing Page 20
Processing Page 21
Processing Page 22
Processing Page 23
Processing Page 24
Processing Page 25
Processing Page 26
Processing Page 27
Processing Page 28
Processing Page 29
Processing Page 30
Processing Page 31
Processing Page 32
Processing Page 33
Processing Page 34
Processing Page 35
Processing Page 36
Processing Page 37
Processing Page 38
Processing Page 39
Processing Page 40
Processing Page 41
Processing Page 42
Processing Page 43
Processing Page 44
Processing Page 45
Processing Page 46
Processing Page 47
Processing Page 48
Processing Page 49
Processing Page 50
Processing Page 51
Processing Page 52
Processing Page 53
Pr

In [17]:
# CREATE DATAFRAME
df = pd.DataFrame(
    items,
    columns=[
        "Product Name",
        "Brand",
        "Price",
        "Original Price",
        "Discount %",
        "Rating",
        "Ratings Count",
        "Reviews Count",
        "RAM (GB)",
        "Storage (GB)",
        "Rear Camera (MP)",
        "Front Camera (MP)",
        "Battery (mAh)"
    ]
)

df.head()

,Product Name,Brand,Price,Original Price,Discount %,Rating,Ratings Count,Reviews Count,RAM (GB),Storage (GB),Rear Camera (MP),Front Camera (MP),Battery (mAh)
0,"OPPO K14x 5G (Icy Blue, 128 GB)",OPPO,16999,17999,5,4.3,3075,198,4,128,50MP + 2MP,5MP; f/2.2; FOV 77°; 3P lens,6500
1,"Samsung M06 5G (Sage Green, 128 GB)",Samsung,12690,15499,18,4.1,1999,127,4,128,50MP Rear Camera,None,5000
2,"MOTOROLA g37 (PANTONE Impenetrable, 64 GB)",MOTOROLA,18999,24999,24,4,11,2,4,64,50MP Rear Camera,8MP Front Camera,5200
3,"Samsung M06 5G (Sage Green, 128 GB)",Samsung,14168,18999,25,4.1,1502,83,6,128,50MP Rear Camera,None,5000
4,"POCO C85x Charger in the Box (Emerald Green, 6...",POCO,12499,20999,40,4.1,1662,92,4,64,32MP Rear Camera,None,6300


In [19]:
# SAVE FILES
df.to_csv(
    "smartphones_raw.csv",
    index=False
)

print(f"Total Phones = {len(df)}")
print("Dataset saved successfully")

Total Phones = 984
Dataset saved successfully
